<a href="https://colab.research.google.com/github/MamushiSushi/Crotalus_tail_banding_ratios/blob/main/ColabNotebook/tailband_colab_region_Mod.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Rattlesnake tail-band project: Colab runner

Runs the photo download and blinding steps in Colab, saving everything to your Google Drive in **MyDrive/rattlesnake-tailbands/**.

Run the cells top to bottom. Steps 1–3 are needed every time you open the notebook.

## 1. Connect Google Drive and go to the project folder

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT = '/content/drive/MyDrive/rattlesnake-tailbands'
os.makedirs(PROJECT, exist_ok=True)
os.chdir(PROJECT)
print('Working in', os.getcwd())

Mounted at /content/drive
Working in /content/drive/MyDrive/rattlesnake-tailbands


## 2. Install the iNaturalist package

In [2]:
!pip install -q pyinaturalist

Check Version:

In [ ]:
from importlib.metadata import version

print(version("pyinaturalist"))

0.21.1


## 3. Save the two scripts into the project folder

In [ ]:
%%writefile inat_pull.py
"""
Pull CC-licensed, research-grade iNaturalist photos of Crotalus atrox and C. scutulatus.

Step 1 builds a metadata table with ONE ROW PER PHOTO (not per observation),
because tail close-ups are often the 2nd-4th photo of an observation.
Step 2 downloads photos from the iNaturalist open-data bucket (licensed photos only).

Run in Colab or on Bridges-2:
    pip install pyinaturalist pandas requests
    python inat_pull.py metadata                 # -> inat_photos.csv (all localities)
    python inat_pull.py metadata --place-id 1234 # one iNat place only (ID from an Explore URL)
    python inat_pull.py metadata --bbox 29 -105 32 -103   # or a custom box: SW lat/lng, NE lat/lng
    python inat_pull.py download --n 1500        # random sample per species -> photos/<species>/
"""
import argparse
import os
import time

import pandas as pd
import requests
from pyinaturalist import get_observations  # handles paging + iNat rate limits

SPECIES = ["Crotalus atrox", "Crotalus scutulatus"]
LICENSES = ["CC0", "CC-BY", "CC-BY-NC", "CC-BY-SA", "CC-BY-NC-SA", "CC-BY-ND", "CC-BY-NC-ND"]
META = "inat_photos.csv"


def pull_metadata(place_id=None, bbox=None):
    region = {}
    if place_id:
        region["place_id"] = place_id
    if bbox:
        region.update(dict(zip(["swlat", "swlng", "nelat", "nelng"], bbox)))
    print("Region filter:", region or "none (all localities)")
    rows = []
    for sp in SPECIES:
        print(f"Querying {sp} ...")
        resp = get_observations(
            taxon_name=sp,
            quality_grade="research",
            photo_license=LICENSES,
            photos=True,
            page="all",
            **region,
        )
        obs_list = resp["results"]
        print(f"  {len(obs_list)} observations")
        for o in obs_list:
            # pyinaturalist returns location as [lat, lng]; the raw API returns "lat,lng"
            loc = o.get("location")
            if isinstance(loc, str) and loc:
                lat, lon = (float(x) for x in loc.split(","))
            elif isinstance(loc, (list, tuple)) and len(loc) == 2:
                lat, lon = float(loc[0]), float(loc[1])
            else:
                lat, lon = None, None
            for i, p in enumerate(o.get("photos", [])):
                if not p.get("license_code"):
                    continue  # unlicensed photo inside a licensed observation: skip
                rows.append(
                    {
                        "species": sp,
                        "obs_id": o["id"],
                        "photo_id": p["id"],
                        "photo_index": i,  # 0 = first photo of the observation
                        "n_photos_in_obs": len(o.get("photos", [])),
                        "license": p["license_code"],
                        "attribution": p.get("attribution"),
                        "url_large": p["url"].replace("square", "large"),
                        "lat": lat,
                        "lon": lon,
                        "coords_obscured": o.get("obscured"),
                        "positional_accuracy_m": o.get("positional_accuracy"),
                        "observed_on": o.get("observed_on"),
                        "observer": (o.get("user") or {}).get("login"),
                        "obs_url": f"https://www.inaturalist.org/observations/{o['id']}",
                    }
                )
    df = pd.DataFrame(rows)
    df.to_csv(META, index=False)
    print(df.groupby("species").agg(photos=("photo_id", "size"), observations=("obs_id", "nunique")))
    print(f"Saved {META}")


def download(n_per_species, seed=42):
    df = pd.read_csv(META)
    for sp, grp in df.groupby("species"):
        # sample observations (not photos) so one snake can't dominate, then keep all its photos
        obs = grp["obs_id"].drop_duplicates().sample(
            n=min(n_per_species, grp["obs_id"].nunique()), random_state=seed
        )
        sub = grp[grp["obs_id"].isin(obs)]
        outdir = os.path.join("photos", sp.replace(" ", "_"))
        os.makedirs(outdir, exist_ok=True)
        print(f"{sp}: downloading {len(sub)} photos from {len(obs)} observations")
        for _, r in sub.iterrows():
            ext = os.path.splitext(r["url_large"].split("?")[0])[1] or ".jpg"  # drop ?timestamp
            path = os.path.join(outdir, f"{r['obs_id']}_{r['photo_id']}{ext}")
            if os.path.exists(path):
                continue
            try:
                resp = requests.get(r["url_large"], timeout=30)
                resp.raise_for_status()
                with open(path, "wb") as f:
                    f.write(resp.content)
            except Exception as e:  # keep going; log failures
                print("  failed:", r["url_large"], e)
            time.sleep(0.2)  # be polite even though the open-data bucket has no download cap


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("step", choices=["metadata", "download"])
    ap.add_argument("--n", type=int, default=1500, help="observations to sample per species")
    ap.add_argument("--place-id", type=int, default=None,
                    help="iNaturalist place ID (from the place_id= number in an Explore URL)")
    ap.add_argument("--bbox", type=float, nargs=4, default=None,
                    metavar=("SWLAT", "SWLNG", "NELAT", "NELNG"),
                    help="custom bounding box instead of a named place")
    a = ap.parse_args()
    if a.step == "metadata":
        pull_metadata(a.place_id, a.bbox)
    else:
        download(a.n)

Overwriting inat_pull.py


In [ ]:
%%writefile blind.py
"""
Blind tail crops for measurement.

Expected input layout (crop each photo to the tail only first):
    crops/
      measurement/Crotalus_atrox/*.jpg
      measurement/Crotalus_scutulatus/*.jpg
      test/Crotalus_atrox/*.jpg
      test/Crotalus_scutulatus/*.jpg

Usage:
    python blind.py round1                      # all crops -> blind/round1/  (codes A001...)
    python blind.py round2                      # all crops again, NEW codes  (codes B001...)
    python blind.py round2 --fraction 0.5       # or a random half

Outputs:
    blind/roundN/<code>.jpg          images to measure (no species, set, or obs ID in the name)
    blind/roundN_measurements.csv    sheet to fill in, rows already in random order
    keys/key_roundN.csv              code -> species/set/original file  (DON'T OPEN until done)

Measure in the order the sheet lists. In Fiji: straight-line tool, press M for each band
(light1, dark1, light2, dark2, light3, dark3 from the rattle forward), copy the lengths into
the sheet. Units don't matter (pixels are fine) because only the ratio is used.
"""
import argparse
import csv
import os
import random
import shutil

EXTS = (".jpg", ".jpeg", ".png", ".tif", ".tiff")
COLS = ["code", "L1", "D1", "L2", "D2", "L3", "D3", "excluded", "exclusion_reason", "notes"]


def collect(root):
    items = []
    for set_name in sorted(os.listdir(root)):
        set_dir = os.path.join(root, set_name)
        if not os.path.isdir(set_dir):
            continue
        for species in sorted(os.listdir(set_dir)):
            sp_dir = os.path.join(set_dir, species)
            if not os.path.isdir(sp_dir):
                continue
            for f in sorted(os.listdir(sp_dir)):
                if f.lower().endswith(EXTS):
                    items.append({"set": set_name, "species": species.replace("_", " "),
                                  "original_file": os.path.join(sp_dir, f)})
    return items


def blind(round_name, root, fraction, seed):
    items = collect(root)
    if not items:
        raise SystemExit(f"No images found under {root}/<set>/<species>/")
    rng = random.Random(seed)
    if fraction < 1:
        items = rng.sample(items, k=max(1, round(len(items) * fraction)))
    rng.shuffle(items)

    prefix = {"round1": "A", "round2": "B"}.get(round_name, "C")
    outdir = os.path.join("blind", round_name)
    if os.path.exists(outdir):
        raise SystemExit(f"{outdir} already exists - delete it first if you really want to redo this round")
    os.makedirs(outdir)
    os.makedirs("keys", exist_ok=True)

    key_rows = []
    for i, it in enumerate(items, 1):
        code = f"{prefix}{i:03d}"
        ext = os.path.splitext(it["original_file"])[1].lower()
        shutil.copy2(it["original_file"], os.path.join(outdir, code + ext))
        key_rows.append({"code": code, **it})

    # Measurement sheet in a SECOND independent random order, so sheet order != code order
    sheet_codes = [r["code"] for r in key_rows]
    rng.shuffle(sheet_codes)
    with open(os.path.join("blind", f"{round_name}_measurements.csv"), "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=COLS)
        w.writeheader()
        for c in sheet_codes:
            w.writerow({"code": c})

    with open(os.path.join("keys", f"key_{round_name}.csv"), "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["code", "set", "species", "original_file"])
        w.writeheader()
        w.writerows(key_rows)

    print(f"{round_name}: {len(key_rows)} images -> {outdir}/")
    print(f"Fill in blind/{round_name}_measurements.csv in the listed order.")
    print(f"Key saved to keys/key_{round_name}.csv - don't open it until measuring is finished.")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("round", choices=["round1", "round2", "round3"])
    ap.add_argument("--crops", default="crops")
    ap.add_argument("--fraction", type=float, default=1.0)
    ap.add_argument("--seed", type=int, default=None, help="leave unset for an unpredictable order")
    a = ap.parse_args()
    blind(a.round, a.crops, a.fraction, a.seed)


Overwriting blind.py


## 4. Pull metadata (one row per photo)
Creates `inat_photos.csv`. Check the species counts it prints.

**To limit to one region:** on iNaturalist Explore, pick the place, then copy the number after `place_id=` in the page URL and put it in `PLACE_ID` below. Leave `PLACE_ID = None` for all localities. (A custom box also works: `--bbox SWLAT SWLNG NELAT NELNG`.)

Note: for brevity Trans-Pecos Ecoregion (place_id=79259) was selected as all areas with C. scutulatus also include C. atrox

In [ ]:
PLACE_ID = 79259   # e.g. 1234 - replace with the number from your Explore URL

if PLACE_ID:
    !python inat_pull.py metadata --place-id {PLACE_ID}
else:
    !python inat_pull.py metadata

python3: can't open file '/content/inat_pull.py': [Errno 2] No such file or directory


## 5. Download a sample of photos
Creates `photos/Crotalus_atrox/` and `photos/Crotalus_scutulatus/` in your Drive. Raise or lower `--n` (observations per species) as needed.

In [ ]:
!python inat_pull.py download --n 600

Crotalus atrox: downloading 888 photos from 600 observations
Crotalus scutulatus: downloading 904 photos from 600 observations


## 6. Screen and crop (outside Colab)
1. Get the `photos/` folder onto your laptop: download it from drive.google.com, or use the Google Drive for desktop app so it syncs automatically.
2. In **Fiji** (standalone app, no Python needed), crop each usable photo to the tail only.
3. Save the crops into this layout inside the Drive project folder:
```
crops/measurement/Crotalus_atrox/
crops/measurement/Crotalus_scutulatus/
crops/test/Crotalus_atrox/
crops/test/Crotalus_scutulatus/
```
Then come back and run step 7.

## 7. Blind Round 1
Creates `blind/round1/` (coded images), `blind/round1_measurements.csv` (your measurement sheet) and `keys/key_round1.csv`. Don't open the key until measuring is finished.

In [ ]:
!python blind.py round1

round1: 200 images -> blind/round1/
Fill in blind/round1_measurements.csv in the listed order.
Key saved to keys/key_round1.csv - don't open it until measuring is finished.


## 8. Download the blinded images as a zip
Or just open `blind/round1/` through Google Drive for desktop.

In [ ]:
import shutil
shutil.make_archive('blind_round1', 'zip', 'blind/round1')
from google.colab import files
files.download('blind_round1.zip')

## 9. Round 2 (at least 3 days after finishing Round 1)
New codes and a new random order. Use `--fraction 0.5` to re-measure a random half instead of everything.

In [ ]:
!python blind.py round2

round2: 200 images -> blind/round2/
Fill in blind/round2_measurements.csv in the listed order.
Key saved to keys/key_round2.csv - don't open it until measuring is finished.


# Ensure no duplicate Individuals across test and measurement sets.

In [ ]:
import os, pandas as pd

rows = []
for root, _, files in os.walk("crops"):
    for f in files:
        if f.lower().endswith((".jpg", ".jpeg", ".png")):
            parts = root.split(os.sep)            # crops / set / species
            rows.append({"set": parts[1], "species": parts[2], "file": f,
                         "obs_id": f.split("_")[0]})
df = pd.DataFrame(rows)

print(df.groupby(["set", "species"]).size().to_string(), "\n")

bad = df[~df["obs_id"].str.isdigit()]
if len(bad):
    print("Files not named observationID_photoID:\n", bad[["set", "species", "file"]].to_string(index=False), "\n")

dups = df[df.duplicated("obs_id", keep=False)].sort_values("obs_id")
if len(dups):
    print(f"DUPLICATES: {dups['obs_id'].nunique()} snakes appear more than once:")
    print(dups[["obs_id", "set", "species", "file"]].to_string(index=False))
    print("\nKeep one photo per obs_id and remove the others.")
else:
    print("No duplicates: every snake appears once.")

set          species            
measurement  Crotalus_atrox         75
             Crotalus_scutulatus    75
test         Crotalus_atrox         25
             Crotalus_scutulatus    25 

No duplicates: every snake appears once.


In [ ]:
import requests; print(requests.__version__)

2.32.4


#Final Analysis: Round 1 and 2 Blin repeat of Tail0band ratios.

In [8]:
import os, shutil
from google.colab import drive, files
drive.mount('/content/drive')
work = '/content/drive/MyDrive/rattlesnake-tailbands/analysis'
os.makedirs(work, exist_ok=True)
os.chdir(work)

# copy the two keys from your keys folder
shutil.copy('../keys/key_round1.csv', 'key_round1.csv')
shutil.copy('../keys/key_round2.csv', 'key_round2.csv')

print("Now select your 2 Fiji results files and 2 exclusion files")
up = files.upload()

rename = {}
for f in up:
    low = f.lower()
    if 'exclu' in low and '1' in low: rename[f] = 'Round_1_Photo_Exclusions.md'
    elif 'exclu' in low and '2' in low: rename[f] = 'Round_2_Photo_Exclusions.txt'
    elif '1' in low: rename[f] = 'round1.csv'
    elif '2' in low: rename[f] = 'round2.csv'
for old, new in rename.items():
    if old != new: os.replace(old, new)
    print(f"{old}  ->  {new}")
print(sorted(os.listdir('.')))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Now select your 2 Fiji results files and 2 exclusion files


Saving Round_1_Photo_Exclusions.txt to Round_1_Photo_Exclusions.txt
Saving Round_2_Photo_Exclusions.txt to Round_2_Photo_Exclusions.txt
Saving round1.cvs to round1.cvs
Saving round2.cvs to round2.cvs
Round_1_Photo_Exclusions.txt  ->  Round_1_Photo_Exclusions.md
Round_2_Photo_Exclusions.txt  ->  Round_2_Photo_Exclusions.txt
round1.cvs  ->  round1.csv
round2.cvs  ->  round2.csv
['Round_1_Photo_Exclusions.md', 'Round_2_Photo_Exclusions.txt', 'key_round1.csv', 'key_round2.csv', 'round1.csv', 'round2.csv']


In [9]:
"""
Final analysis: Round 1 + Round 2 (blinded repeat) tail-band light:dark ratios,
Crotalus atrox vs C. scutulatus.

Inputs: round1.csv, key_round1.csv, Round_1_Photo_Exclusions.md,
        round2.csv, key_round2.csv, Round_2_Photo_Exclusions.txt
Rules: light bands measured first, then dark (3+3 or 2+2); ratio = mean light / mean dark.
Round 1 corrections (documented): A038 kept, A039 excluded; A022 = 3 light + 2 dark;
A044 = 2+2 (extra 5th row dropped); A015, A017 measured alternating light/dark.

Outputs: final_individuals.csv, final_summary.txt,
         fig1_ratio_by_species.png, fig2_roc.png, fig3_bland_altman.png, fig4_round1_vs_round2.png
"""
import re
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker

POS, NEG = "Crotalus scutulatus", "Crotalus atrox"
rng = np.random.default_rng(42)
out = []
def p(s=""): out.append(str(s)); print(s)

# ---------------- per-round ratios ----------------
SPECIAL = {"A022": ([0, 1, 2], [3, 4]), "A044": ([0, 1], [2, 3]),
           "A015": ([0, 2], [1, 3]), "A017": ([0, 2], [1, 3])}

def ratios(res_file, key_file, exc_file, prefix, fix_exc=None):
    res = pd.read_csv(res_file, sep=None, engine="python")
    res.columns = [c.strip() for c in res.columns]
    res["code"] = res["Label"].str.extract(rf"({prefix}\d{{3}})")
    exc = set(re.findall(rf"{prefix}\d{{3}}", open(exc_file, encoding="utf-8").read()))
    if fix_exc: exc = fix_exc(exc)
    rows = []
    for code, g in res.groupby("code"):
        L = g["Length"].to_numpy(); n = len(L)
        if code in SPECIAL: li, di = SPECIAL[code]
        elif n in (4, 6): li, di = list(range(n // 2)), list(range(n // 2, n))
        else: continue
        rows.append({"code": code, "light": L[li].mean(), "dark": L[di].mean(),
                     "ratio": L[li].mean() / L[di].mean(), "nb": len(di)})
    d = pd.DataFrame(rows)
    key = pd.read_csv(key_file)
    d = key.merge(d, on="code", how="left")
    d["excluded"] = d["code"].isin(exc)
    d.loc[d.excluded, ["light", "dark", "ratio"]] = np.nan
    return d

r1 = ratios("round1.csv", "key_round1.csv", "Round_1_Photo_Exclusions.md", "A",
            lambda e: (e - {"A038"}) | {"A039"})
r2 = ratios("round2.csv", "key_round2.csv", "Round_2_Photo_Exclusions.txt", "B")
m = r1.merge(r2, on=["original_file", "set", "species"], suffixes=("_1", "_2"))
assert len(m) == 200
m["y"] = (m["species"] == POS).astype(int)

# ---------------- exclusion agreement ----------------
p("FINAL SUMMARY: Round 1 + Round 2")
e1, e2 = m.excluded_1, m.excluded_2
tab = pd.crosstab(e1.rename("Excluded R1"), e2.rename("Excluded R2"))
p("Exclusion agreement (200 individuals):"); p(tab.to_string())
po = (e1 == e2).mean()
pe = e1.mean() * e2.mean() + (1 - e1.mean()) * (1 - e2.mean())
p(f"Observed agreement {po:.1%}; Cohen's kappa = {(po - pe) / (1 - pe):.2f}")
both = m[~e1 & ~e2].copy()
p(f"Measured in both rounds: {len(both)}; excluded in at least one round: {200 - len(both)}")
p(pd.crosstab(both.set, both.species).to_string()); p()

# ---------------- repeatability ----------------
def icc_a1(x1, x2):
    """ICC(2,1): two-way random effects, absolute agreement, single measurement."""
    Y = np.c_[x1, x2]; n, k = Y.shape
    gm = Y.mean()
    msr = k * ((Y.mean(1) - gm) ** 2).sum() / (n - 1)
    msc = n * ((Y.mean(0) - gm) ** 2).sum() / (k - 1)
    sse = ((Y - Y.mean(1, keepdims=True) - Y.mean(0, keepdims=True) + gm) ** 2).sum()
    mse = sse / ((n - 1) * (k - 1))
    return (msr - mse) / (msr + (k - 1) * mse + k * (msc - mse) / n)

def boot_icc(x1, x2, B=2000):
    idx = [rng.integers(0, len(x1), len(x1)) for _ in range(B)]
    v = [icc_a1(x1[i], x2[i]) for i in idx]
    return np.percentile(v, [2.5, 97.5])

p("Repeatability (individuals measured in both rounds)")
for lab, a, b in [("ratio", both.ratio_1.values, both.ratio_2.values),
                  ("ln(ratio)", np.log(both.ratio_1.values), np.log(both.ratio_2.values)),
                  ("mean light width (px)", both.light_1.values, both.light_2.values),
                  ("mean dark width (px)", both.dark_1.values, both.dark_2.values)]:
    ci = boot_icc(a, b)
    p(f"  ICC(2,1) {lab:22s} = {icc_a1(a, b):.3f}  (95% CI {ci[0]:.3f}-{ci[1]:.3f})")
diff = both.ratio_2 - both.ratio_1
mean_r = (both.ratio_1 + both.ratio_2) / 2
bias, sd = diff.mean(), diff.std(ddof=1)
t = stats.ttest_rel(both.ratio_2, both.ratio_1)
p(f"  Bland-Altman (ratio): bias R2-R1 = {bias:+.3f} (paired t p={t.pvalue:.2f}); "
  f"95% limits of agreement {bias - 1.96 * sd:+.3f} to {bias + 1.96 * sd:+.3f}")
p(f"  Median absolute difference between rounds = {diff.abs().median():.3f}; "
  f"as % of the individual's mean ratio = {(diff.abs() / mean_r).median():.1%}")
p(f"  Spearman r (R1 vs R2) = {stats.spearmanr(both.ratio_1, both.ratio_2)[0]:.3f}")

# ---------------- classification ----------------
def fit_threshold(train):
    X = np.log(train[["r"]].to_numpy()); y = train.y.to_numpy()
    lr = LogisticRegression(C=1e6).fit(X, y)
    b0, b1 = lr.intercept_[0], lr.coef_[0, 0]
    fpr, tpr, thr = roc_curve(y, lr.predict_proba(X)[:, 1])
    pc = thr[np.argmax(tpr - fpr)]
    return float(np.exp((np.log(pc / (1 - pc)) - b0) / b1)), b0, b1

def metrics(y, r, cut):
    pred = (r >= cut).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    se, sp = tp / (tp + fn), tn / (tn + fp)
    return dict(bacc=(se + sp) / 2, sens=se, spec=sp, auc=roc_auc_score(y, r), cm=(tn, fp, fn, tp))

def boot(sub, cut, B=2000):
    y, r = sub.y.to_numpy(), sub.r.to_numpy()
    i0, i1 = np.where(y == 0)[0], np.where(y == 1)[0]
    v = {k: [] for k in ("bacc", "sens", "spec", "auc")}
    for _ in range(B):
        idx = np.r_[rng.choice(i0, len(i0)), rng.choice(i1, len(i1))]
        mm = metrics(y[idx], r[idx], cut)
        for k in v: v[k].append(mm[k])
    return {k: np.percentile(x, [2.5, 97.5]) for k, x in v.items()}

def evaluate(label, data):
    tr, te = data[data.set == "measurement"], data[data.set == "test"]
    cut, b0, b1 = fit_threshold(tr)
    p(f"\n{label}: measurement n={len(tr)}, test n={len(te)}; fitted threshold ratio >= {cut:.2f} -> scutulatus")
    res = {}
    for nm, sub in [("Measurement (in-sample)", tr), ("TEST (independent)", te)]:
        mm, ci = metrics(sub.y.to_numpy(), sub.r.to_numpy(), cut), boot(sub, cut)
        tn, fp, fn, tp = mm["cm"]
        p(f"  {nm}: balanced acc {mm['bacc']:.3f} ({ci['bacc'][0]:.3f}-{ci['bacc'][1]:.3f}), "
          f"AUC {mm['auc']:.3f} ({ci['auc'][0]:.3f}-{ci['auc'][1]:.3f}), "
          f"sens {mm['sens']:.2f}, spec {mm['spec']:.2f}  "
          f"[atrox ok {tn}, atrox->scut {fp}, scut->atrox {fn}, scut ok {tp}]")
        res[nm] = (mm, ci)
    return cut, res

# Primary: mean of the two rounds, individuals measured in both rounds
both["r"] = (both.ratio_1 + both.ratio_2) / 2
cut_main, res_main = evaluate("PRIMARY - mean of both rounds", both)
r1o = m[~e1].assign(r=lambda d: d.ratio_1)
r2o = m[~e2].assign(r=lambda d: d.ratio_2)
cut_r1, _ = evaluate("Round 1 only (all R1-measured)", r1o)
cut_r2, _ = evaluate("Round 2 only (all R2-measured)", r2o)

# Class changes between rounds (same threshold applied to both rounds)
for lab, c in [("primary threshold", cut_main), ("Round 1 threshold", cut_r1)]:
    flip = ((both.ratio_1 >= c) != (both.ratio_2 >= c)).mean()
    p(f"\nIndividuals whose predicted species changes between rounds ({lab} {c:.2f}): {flip:.1%}")

# Descriptives and species difference on primary values
p("\nLight:dark ratio (mean of both rounds) by set and species")
p(both.groupby(["set", "species"])["r"].describe()[["count", "mean", "std", "min", "50%", "max"]].round(2).to_string())
for nm, sub in [("measurement", both[both.set == "measurement"]), ("test", both[both.set == "test"]), ("all", both)]:
    a, b = sub.loc[sub.y == 0, "r"], sub.loc[sub.y == 1, "r"]
    u = stats.mannwhitneyu(b, a, alternative="two-sided")
    p(f"Mann-Whitney ({nm}): scutulatus median {b.median():.2f} vs atrox {a.median():.2f}, U={u.statistic:.0f}, p={u.pvalue:.2g}")
p(f"scutulatus with ratio >= 2.0: {(both.loc[both.y == 1, 'r'] >= 2).mean():.0%}; "
  f"atrox within 0.8-1.25: {both.loc[both.y == 0, 'r'].between(0.8, 1.25).mean():.0%}")
big = both[(both.dark_1 + both.dark_2) / 2 >= 8]
te_big = big[big.set == "test"]
p(f"Robustness: tails with mean dark band >= 8 px (n={len(big)}): test AUC = "
  f"{roc_auc_score(te_big.y, te_big.r):.3f} (test n={len(te_big)})")

cols = ["original_file", "set", "species", "code_1", "code_2", "excluded_1", "excluded_2",
        "ratio_1", "ratio_2", "light_1", "dark_1", "light_2", "dark_2"]
m[cols].to_csv("final_individuals.csv", index=False)
open("final_summary.txt", "w").write("\n".join(out))

# ---------------- figures ----------------
col = {NEG: "#4C72B0", POS: "#DD8452"}
fig, ax = plt.subplots(figsize=(7.5, 4.8))
for k, (sp, lab) in enumerate([(NEG, "C. atrox"), (POS, "C. scutulatus")]):
    for s, mk in [("measurement", "o"), ("test", "^")]:
        v = both[(both.species == sp) & (both.set == s)]["r"]
        ax.scatter(v, k + rng.uniform(-0.18, 0.18, len(v)), s=28, marker=mk, alpha=0.75, color=col[sp],
                   edgecolor="white" if mk == "o" else "black", linewidth=0.6, label=f"{lab}, {s} set (n={len(v)})")
    med = both[both.species == sp]["r"].median()
    ax.plot([med] * 2, [k - 0.3, k + 0.3], color="black", lw=2)
ax.axvline(cut_main, color="crimson", lw=1.6, ls="--", label=f"Fitted threshold = {cut_main:.2f}")
for xv, tt in [(1, "1:1"), (2, "2:1")]:
    ax.axvline(xv, color="grey", lw=0.8, ls=":"); ax.text(xv, 1.5, tt, ha="center", color="grey", fontsize=9)
ax.set_xscale("log"); ax.set_xticks([0.75, 1, 1.5, 2, 3, 4]); ax.set_xticklabels(["0.75", "1", "1.5", "2", "3", "4"]); ax.xaxis.set_minor_formatter(matplotlib.ticker.NullFormatter())
ax.set_yticks([0, 1]); ax.set_yticklabels(["C. atrox", "C. scutulatus"], style="italic"); ax.set_ylim(-0.5, 1.6)
ax.set_xlabel("Light:dark tail-band width ratio, mean of two blinded rounds (log scale)")
ax.legend(fontsize=7.5, loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=3, frameon=False)
ax.spines[["top", "right"]].set_visible(False); plt.tight_layout(); plt.savefig("fig1_ratio_by_species.png", dpi=200)

fig, ax = plt.subplots(figsize=(4.2, 4))
for nm, sub, c in [("Measurement", both[both.set == "measurement"], "grey"), ("Test", both[both.set == "test"], "crimson")]:
    f, tp_, _ = roc_curve(sub.y, sub.r)
    ax.plot(f, tp_, color=c, lw=2, label=f"{nm} (AUC={roc_auc_score(sub.y, sub.r):.2f})")
ax.plot([0, 1], [0, 1], ":", color="black", lw=0.8)
ax.set_xlabel("False-positive rate (atrox called scutulatus)"); ax.set_ylabel("True-positive rate")
ax.legend(frameon=False, fontsize=9); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.savefig("fig2_roc.png", dpi=200)

fig, ax = plt.subplots(figsize=(5.5, 4))
ax.scatter(mean_r, diff, s=20, alpha=0.7, color="#555555", edgecolor="white", linewidth=0.5)
for yv, ls in [(bias, "-"), (bias - 1.96 * sd, "--"), (bias + 1.96 * sd, "--")]:
    ax.axhline(yv, color="crimson", ls=ls, lw=1.3)
ax.axhline(0, color="black", lw=0.6, ls=":")
ax.text(ax.get_xlim()[1], bias + 1.96 * sd, f" +1.96 SD ({bias + 1.96 * sd:+.2f})", va="bottom", ha="right", fontsize=8, color="crimson")
ax.text(ax.get_xlim()[1], bias - 1.96 * sd, f" -1.96 SD ({bias - 1.96 * sd:+.2f})", va="top", ha="right", fontsize=8, color="crimson")
ax.set_xlabel("Mean ratio of the two rounds"); ax.set_ylabel("Round 2 − Round 1 ratio")
ax.spines[["top", "right"]].set_visible(False); plt.tight_layout(); plt.savefig("fig3_bland_altman.png", dpi=200)

fig, ax = plt.subplots(figsize=(4.5, 4.5))
for sp, lab in [(NEG, "C. atrox"), (POS, "C. scutulatus")]:
    s = both[both.species == sp]
    ax.scatter(s.ratio_1, s.ratio_2, s=20, alpha=0.75, color=col[sp], edgecolor="white", linewidth=0.5, label=lab)
lim = [0.5, max(both.ratio_1.max(), both.ratio_2.max()) * 1.05]
ax.plot(lim, lim, color="black", lw=0.8, ls=":")
ax.axvline(cut_main, color="crimson", lw=0.8, ls="--"); ax.axhline(cut_main, color="crimson", lw=0.8, ls="--")
ax.set_xlim(lim); ax.set_ylim(lim); ax.set_xlabel("Round 1 ratio"); ax.set_ylabel("Round 2 ratio")
ax.legend(frameon=False, fontsize=9, prop={"style": "italic"}); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.savefig("fig4_round1_vs_round2.png", dpi=200)

FINAL SUMMARY: Round 1 + Round 2
Exclusion agreement (200 individuals):
Excluded R2  False  True 
Excluded R1              
False          169     12
True             3     16
Observed agreement 92.5%; Cohen's kappa = 0.64
Measured in both rounds: 169; excluded in at least one round: 31
species      Crotalus atrox  Crotalus scutulatus
set                                             
measurement              62                   62
test                     22                   23

Repeatability (individuals measured in both rounds)
  ICC(2,1) ratio                  = 0.656  (95% CI 0.547-0.741)
  ICC(2,1) ln(ratio)              = 0.659  (95% CI 0.559-0.743)
  ICC(2,1) mean light width (px)  = 0.969  (95% CI 0.945-0.984)
  ICC(2,1) mean dark width (px)   = 0.973  (95% CI 0.957-0.983)
  Bland-Altman (ratio): bias R2-R1 = -0.089 (paired t p=0.00); 95% limits of agreement -0.675 to +0.498
  Median absolute difference between rounds = 0.154; as % of the individual's mean ratio = 14.1%
  Spea